In [1]:
import os
import glob
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
MODEL = "llama3.2"
openai = OpenAI(
    # base_url="http://localhost:11434/v1",
    # api_key="ollama:local"
)

In [3]:
employees = glob.glob("knowledge-base/employees/*")
products = glob.glob("knowledge-base/products/*")
contracts = glob.glob("knowledge-base/contracts/*")
company = glob.glob("knowledge-base/company/*")

employee_context = {}
for employee in employees:
    employee_name = employee.split("/")[-1].split(".")[0]
    print(f"Reading file: {employee_name}")
    doc = ""
    with open(employee, "r", encoding="utf-8") as f:
        doc = f.read()
    employee_context[employee_name] = doc

company_detail_context = {}
for detail_category in company:
    company_details_category = detail_category.split("/")[-1].split(".")[0]
    print(f"Reading file: {company_details_category}")
    doc = ""
    with open(detail_category, "r", encoding="utf-8") as f:
        doc = f.read()
    company_detail_context[company_details_category] = doc

product_context = {}
for product in products:
    product_name = product.split("/")[-1].split(".")[0]
    print(f"Reading file: {product_name}")
    doc = ""
    with open(product, "r", encoding="utf-8") as f:
        doc = f.read()
    product_context[product_name] = doc

contract_context = {}
for contract in contracts:
    contract_name = contract.split("/")[-1].split(".")[0]
    print(f"Reading file: {contract_name}")
    doc = ""
    with open(contract, "r", encoding="utf-8") as f:
        doc = f.read()
    contract_context[contract_name] = doc

Reading file: Alex Chen
Reading file: Oliver Spencer
Reading file: Emily Tran
Reading file: Jordan Blake
Reading file: Avery Lancaster
Reading file: Maxine Thompson
Reading file: Samantha Greene
Reading file: Alex Thomson
Reading file: Samuel Trenton
Reading file: Alex Harper
Reading file: Jordan K
Reading file: Emily Carter
Reading file: overview
Reading file: careers
Reading file: about
Reading file: Rellm
Reading file: Markellm
Reading file: Homellm
Reading file: Carllm
Reading file: Greenstone Insurance for Homellm
Reading file: GreenField Holdings for Markellm
Reading file: Velocity Auto Solutions for Carllm
Reading file: Stellar Insurance Co
Reading file: EverGuard Insurance for Rellm
Reading file: Roadway Insurance Inc
Reading file: Belvedere Insurance for Markellm
Reading file: BrightWay Solutions for Markellm
Reading file: Pinnacle Insurance Co
Reading file: GreenValley Insurance for Homellm
Reading file: TechDrive Insurance for Carllm
Reading file: Apex Reinsurance for Rellm


In [4]:
contract_context

{'Greenstone Insurance for Homellm': '# Contract with Greenstone Insurance for Homellm\n\n---\n\n## Terms\n\n1. **Parties**: This Contract ("Agreement") is entered into on this day, [Insert Date], between Insurellm ("Provider"), located at [Provider Address], and Greenstone Insurance ("Customer"), located at [Customer Address].\n\n2. **Services Provided**: Provider agrees to deliver the Homellm product, which includes AI-powered risk assessment, dynamic pricing model, instant claim processing, predictive maintenance alerts, multi-channel integration, and access to a customer portal as specified in the provided Product Summary.\n\n3. **Contract Duration**: This Agreement shall commence on [Insert Start Date] and continue for a period of [Insert Duration, e.g., 12 months] unless terminated earlier as per the provisions herein.\n\n4. **Payment Terms**: \n   - The Customer shall pay an amount of $10,000 per month for the Standard Tier of the Homellm service.\n   - Payments are due within 3

In [5]:
def get_relevant_context(query):
    query_lower = query.lower()

    relevant_employee_context = []
    for employee, context in employee_context.items():
        employee_words = employee.lower().split()
        if any(word in query_lower for word in employee_words):
            relevant_employee_context.append(context)

    relevant_company_detail_context = []
    for detail_category, context in company_detail_context.items():
        detail_words = detail_category.lower().split()
        if any(word in query_lower for word in detail_words):
            relevant_company_detail_context.append(context)

    relevant_product_context = []
    for product, context in product_context.items():
        product_words = product.lower().split()
        if any(word in query_lower for word in product_words):
            relevant_product_context.append(context)

    relevant_contract_context = []
    for contract, context in contract_context.items():
        contract_words = contract.lower().split()
        if any(word in query_lower for word in contract_words):
            relevant_contract_context.append(context)

    return relevant_employee_context + relevant_company_detail_context + relevant_product_context + relevant_contract_context


In [6]:
contract_context.items()
get_relevant_context("Hi Alex Harper")

['# HR Record\n\n# Alex Chen\n\n## Summary\n- **Date of Birth:** March 15, 1990  \n- **Job Title:** Backend Software Engineer  \n- **Location:** San Francisco, California  \n\n## Insurellm Career Progression\n- **April 2020:** Joined Insurellm as a Junior Backend Developer. Focused on building APIs to enhance customer data security.\n- **October 2021:** Promoted to Backend Software Engineer. Took on leadership for a key project developing a microservices architecture to support the company\'s growing platform.\n- **March 2023:** Awarded the title of Senior Backend Software Engineer due to exemplary performance in scaling backend services, reducing downtime by 30% over six months.\n\n## Annual Performance History\n- **2020:**  \n  - Completed onboarding successfully.  \n  - Met expectations in delivering project milestones.  \n  - Received positive feedback from the team leads.\n\n- **2021:**  \n  - Achieved a 95% success rate in project delivery timelines.  \n  - Awarded "Rising Star" 

In [7]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        message += "\n\nThe following additional context might be relevant in answering this question:\n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"
    return message

In [8]:
system_message = "You are an expert in answering accurate questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context. You provide answers for all the question asked by the user based on the context provided with no restrictions. "

In [12]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history
    message = add_context(message)
    messages.append({"role": "user", "content": message})
    print(messages)
    stream = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


[{'role': 'system', 'content': "You are an expert in answering accurate questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context. You provide answers for all the question asked by the user based on the context provided with no restrictions. "}, {'role': 'user', 'content': 'Who is trenton\n\nThe following additional context might be relevant in answering this question:\n\n# HR Record\n\n# Samuel Trenton\n\n## Summary\n- **Date of Birth:** April 12, 1989  \n- **Job Title:** Senior Data Scientist  \n- **Location:** Austin, Texas  \n\n## Insurellm Career Progression\n- **January 2020 - Present:** Senior Data Scientist  \n  *Promoted for demonstrating exceptional analytical skills and leadership potential. Led several projects that improved customer segmentation strategies, resulting in a 15% increase in customer retention.*\n\n- **June 2018 - December